In [2]:
import os
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

Phase 3: 

From what I did in the notebook before, add them first into this notebook.

Since we used nrows = 2000000, now we change our test size. 

In [3]:
# Read data
TRAIN_PATH = "train.csv" 
TEST_PATH  = "test.csv"
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

In [4]:
train.shape, test.shape

((55423856, 8), (9914, 7))

In [5]:
# Cleaning set and definition
NYC_BOUNDS = {
    "lat_min": 40.0, "lat_max": 42.0,
    "lon_min": -75.0, "lon_max": -72.0
}

# grid bounds
LAT_MIN, LAT_MAX = 40.5, 41.0
LON_MIN, LON_MAX = -74.3, -73.7

def clean_data(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    x = df.copy()

    # ssential columns
    essential = [
        "pickup_datetime",
        "pickup_longitude", "pickup_latitude",
        "dropoff_longitude", "dropoff_latitude",
        "passenger_count"
    ]
    if is_train:
        essential.append("fare_amount")

    x = x.dropna(subset=essential).copy()

    # datetime
    x["pickup_datetime"] = pd.to_datetime(x["pickup_datetime"], errors="coerce")
    x = x.dropna(subset=["pickup_datetime"]).copy()

    # passenger_count
    x = x[(x["passenger_count"] >= 1) & (x["passenger_count"] <= 6)].copy()

    # geo bounds
    b = NYC_BOUNDS
    geo_keep = (
        x["pickup_latitude"].between(b["lat_min"], b["lat_max"]) &
        x["dropoff_latitude"].between(b["lat_min"], b["lat_max"]) &
        x["pickup_longitude"].between(b["lon_min"], b["lon_max"]) &
        x["dropoff_longitude"].between(b["lon_min"], b["lon_max"])
    )
    x = x.loc[geo_keep].copy()

    # fare_amount
    if is_train:
        x = x[x["fare_amount"] > 0].copy()

    return x

# Sort by time series
def sort_by_time(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.sort_values("pickup_datetime").reset_index(drop=True)
    return out

# add 10*10 grid cell id:
def add_grid(df: pd.DataFrame, lat_col: str, lon_col: str, n_lat: int = 10, n_lon: int = 10, prefix: str = "") -> pd.DataFrame:
    out = df.copy()

    lat = out[lat_col].clip(LAT_MIN, LAT_MAX)
    lon = out[lon_col].clip(LON_MIN, LON_MAX)

    dlat = (LAT_MAX - LAT_MIN) / n_lat
    dlon = (LON_MAX - LON_MIN) / n_lon

    out[f"{prefix}grid_i"] = ((lat - LAT_MIN) / dlat).astype(int).clip(0, n_lat - 1)
    out[f"{prefix}grid_j"] = ((lon - LON_MIN) / dlon).astype(int).clip(0, n_lon - 1)

    out[f"{prefix}cell_id"] = out[f"{prefix}grid_i"] * n_lon + out[f"{prefix}grid_j"]
    return out

# add paired id
def add_pair_id(df: pd.DataFrame, n_cells: int = 100) -> pd.DataFrame:
    out = df.copy()

    # pair_id = pickup_cell_id * 100 + dropoff_cell_id
    out["pair_id"] = out["pu_cell_id"] * n_cells + out["do_cell_id"]

    return out

# return final dataset after grid cell id and paired id
def preprocess_for_pair_grouping(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    out = clean_data(df, is_train=is_train)
    out = sort_by_time(out)

    out = add_grid(out, "pickup_latitude",  "pickup_longitude",  n_lat=10, n_lon=10, prefix="pu_")
    out = add_grid(out, "dropoff_latitude", "dropoff_longitude", n_lat=10, n_lon=10, prefix="do_")

    out = add_pair_id(out, n_cells=100)

    return out

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def fit_xgb_model(X_tr, y_tr):
    model = XGBRegressor(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=12,
        subsample=1.0,
        colsample_bytree=0.8,
        reg_lambda=5.0,
        reg_alpha=0.1,
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror",
        tree_method="hist"
        # device = "cuda"
    )
    model.fit(X_tr, y_tr)
    return model

def rmse_xgboost(X_tr, y_tr, X_val, y_val):
    model = fit_xgb_model(X_tr, y_tr)

    y_tr_pred = model.predict(X_tr)
    y_val_pred = model.predict(X_val)

    train_rmse = rmse(y_tr, y_tr_pred)
    val_rmse = rmse(y_val, y_val_pred)

    print(f"Train RMSE: {train_rmse:.4f}")
    print(f"Val   RMSE: {val_rmse:.4f}")

    return model, train_rmse, val_rmse


def time_series_split(df, train_ratio=0.8):
    """
    按 pickup_datetime 排序
    """
    split_idx = int(len(df) * train_ratio)
    train_part = df.iloc[:split_idx].copy()
    val_part   = df.iloc[split_idx:].copy()
    return train_part, val_part

In [ ]:
train_clean = preprocess_for_pair_grouping(train, is_train=True)

In [ ]:
test_clean = preprocess_for_pair_grouping(test, is_train= False)

In [ ]:
# Grouping result
pair_groups = train_clean.groupby("pair_id")

pair_counts = train_clean["pair_id"].value_counts().sort_values(ascending=False)

print("train_clean shape:", train_clean.shape)
print("test_clean shape :", test_clean.shape)
print("unique pair_id in train:", train_clean["pair_id"].nunique())
print("top 10 pair counts:")
print(pair_counts.head(10))

train_clean shape: (54063069, 15)
test_clean shape : (9914, 14)
unique pair_id in train: 4220
top 10 pair counts:
pair_id
5555    17779906
4545     5958608
4555     5069499
5545     4766409
4544     2316740
4445     2293022
4455     2054155
5544     1964387
4444     1748851
5565      783778
Name: count, dtype: int64


Add the features which already found or did in the previous notbook:

haversine_km : direct distance
manhattan_km : road-like Manhattan distance on lat/lon
trip_bearing: degrees

In [ ]:
# add features definition:
def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0 * c

def manhattan_km(lon1, lat1, lon2, lat2):
    """
    Approximate road-like Manhattan distance on lat/lon:
    north-south leg + east-west leg
    """
    leg1 = haversine_km(lon1, lat1, lon1, lat2)  # move only in latitude
    leg2 = haversine_km(lon1, lat2, lon2, lat2)  # move only in longitude
    return leg1 + leg2

def trip_bearing(lon1, lat1, lon2, lat2):
    """
    Initial bearing (forward azimuth) from pickup to dropoff in degrees.
    Output range: [-180, 180]
    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1

    x = np.sin(dlon) * np.cos(lat2)
    y = (
        np.cos(lat1) * np.sin(lat2)
        - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    )

    bearing = np.degrees(np.arctan2(x, y))
    return bearing

# Define coordinates for points of interest (POIs) in NYC
POI_COORDS = {
    "jfk": (-73.7781, 40.6413),
    "lga": (-73.8740, 40.7769),
    "manhattan_center": (-73.9855, 40.7580),
}

def distance_to_poi(lon, lat, poi_lon, poi_lat):
    return haversine_km(lon, lat, poi_lon, poi_lat)

BASE_YEAR = 2009
BASE_MONTH = 2009 * 12 + 1

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # datetime
    out["pickup_datetime"] = pd.to_datetime(out["pickup_datetime"], errors="coerce", utc=True)
    out["pickup_hour"] = out["pickup_datetime"].dt.hour
    out["pickup_dow"] = out["pickup_datetime"].dt.dayofweek
    out["pickup_month"] = out["pickup_datetime"].dt.month

    # year-related features
    out["pickup_year"] = out["pickup_datetime"].dt.year
    out["year_index"] = out["pickup_year"] - BASE_YEAR
    out["year_month_index"] = (
        out["pickup_datetime"].dt.year * 12 + out["pickup_datetime"].dt.month
        - BASE_MONTH
    )

    out["is_weekend"] = (out["pickup_dow"] >= 5).astype(int)

    # rush hour flags
    out["is_rush_hour"] = out["pickup_hour"].isin([7,8,9,16,17,18,19]).astype(int)
    out["is_night"] = out["pickup_hour"].isin([22,23,0,1,2,3,4,5]).astype(int)

    # distance
    out["haversine_km"] = haversine_km(
        out["pickup_longitude"], out["pickup_latitude"],
        out["dropoff_longitude"], out["dropoff_latitude"]
    )

    out["manhattan_km"] = manhattan_km(
        out["pickup_longitude"], out["pickup_latitude"],
        out["dropoff_longitude"], out["dropoff_latitude"]
    )

    out["distance_ratio_manhattan_haversine"] = (
        out["manhattan_km"] / out["haversine_km"].replace(0, np.nan)
    )

    out["trip_bearing"] = trip_bearing(
        out["pickup_longitude"], out["pickup_latitude"],
        out["dropoff_longitude"], out["dropoff_latitude"]
    )

    bearing_rad = np.radians(out["trip_bearing"])
    out["bearing_sin"] = np.sin(bearing_rad)
    out["bearing_cos"] = np.cos(bearing_rad)

    # POI distance features
    jfk_lon, jfk_lat = POI_COORDS["jfk"]
    lga_lon, lga_lat = POI_COORDS["lga"]
    mh_lon, mh_lat = POI_COORDS["manhattan_center"]

    out["pickup_to_jfk_km"] = distance_to_poi(
        out["pickup_longitude"], out["pickup_latitude"], jfk_lon, jfk_lat
    )
    out["dropoff_to_jfk_km"] = distance_to_poi(
        out["dropoff_longitude"], out["dropoff_latitude"], jfk_lon, jfk_lat
    )

    out["pickup_to_lga_km"] = distance_to_poi(
        out["pickup_longitude"], out["pickup_latitude"], lga_lon, lga_lat
    )
    out["dropoff_to_lga_km"] = distance_to_poi(
        out["dropoff_longitude"], out["dropoff_latitude"], lga_lon, lga_lat
    )

    out["pickup_to_manhattan_km"] = distance_to_poi(
        out["pickup_longitude"], out["pickup_latitude"], mh_lon, mh_lat
    )
    out["dropoff_to_manhattan_km"] = distance_to_poi(
        out["dropoff_longitude"], out["dropoff_latitude"], mh_lon, mh_lat
    )

    # optional airport flags
    airport_radius_km = 2.0

    out["is_jfk_trip"] = (
        (out["pickup_to_jfk_km"] <= airport_radius_km) |
        (out["dropoff_to_jfk_km"] <= airport_radius_km)
    ).astype(int)

    out["is_lga_trip"] = (
        (out["pickup_to_lga_km"] <= airport_radius_km) |
        (out["dropoff_to_lga_km"] <= airport_radius_km)
    ).astype(int)

    out["pickup_in_manhattan_core"] = (
        (out["pickup_longitude"].between(-74.02, -73.93)) &
        (out["pickup_latitude"].between(40.70, 40.82))
    ).astype(int)

    out["dropoff_in_manhattan_core"] = (
        (out["dropoff_longitude"].between(-74.02, -73.93)) &
        (out["dropoff_latitude"].between(40.70, 40.82))
    ).astype(int)

    out["dist_x_jfk"] = out["manhattan_km"] * out["is_jfk_trip"]
    out["dist_x_lga"] = out["manhattan_km"] * out["is_lga_trip"]

    out["is_short_trip"] = (out["manhattan_km"] < 1).astype(int)
    out["is_long_trip"] = (out["manhattan_km"] >= 15).astype(int)
    out["dist_x_long_trip"] = out["manhattan_km"] * out["is_long_trip"]

    out["dist_x_rush"] = out["manhattan_km"] * out["is_rush_hour"]
    out["dist_x_night"] = out["manhattan_km"] * out["is_night"]

    out["hour_sin"] = np.sin(2 * np.pi * out["pickup_hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["pickup_hour"] / 24)

    return out

In [ ]:
train_feat = add_features(train_clean)

In [ ]:
# Settings
MIN_SAMPLES_PER_PAIR = 500

def train_pair_models(train_df, pair_features, target="fare_amount",
                      min_samples_per_pair=MIN_SAMPLES_PER_PAIR,
                      min_train_rows=100,
                      min_val_rows=20):
    """
    返回：
    1. pair_models: dict[pair_id] = trained model
    2. pair_stats_df: 每组训练情况
    3. meta_df: 第二阶段总模型使用的数据（来自各 pair 的 validation 部分）
    """
    pair_models = {}
    pair_stats = []
    meta_parts = []

    grouped = train_df.groupby("pair_id", sort=False)

    for pair_id, g in grouped:
        g = g.sort_values("pickup_datetime").copy()

        n = len(g)
        if n < min_samples_per_pair:
            continue

        g_tr, g_va = time_series_split(g, train_ratio=0.8)

        if len(g_tr) < min_train_rows or len(g_va) < min_val_rows:
            continue

        X_tr = g_tr[pair_features].fillna(0)
        y_tr = g_tr[target]

        X_va = g_va[pair_features].fillna(0)
        y_va = g_va[target]

        model = fit_xgb_model(X_tr, y_tr)
        pred_va = model.predict(X_va)

        pair_models[pair_id] = model

        pair_stats.append({
            "pair_id": pair_id,
            "n_total": len(g),
            "n_train": len(g_tr),
            "n_val": len(g_va),
            "rmse": rmse(y_va.to_numpy(), pred_va),
            "mae": mean_absolute_error(y_va.to_numpy(), pred_va)
        })

        tmp = g_va.copy()
        tmp["pair_pred"] = pred_va
        meta_parts.append(tmp)

    pair_stats_df = pd.DataFrame(pair_stats).sort_values("n_total", ascending=False)

    if len(meta_parts) > 0:
        meta_df = pd.concat(meta_parts, axis=0).sort_values("pickup_datetime").reset_index(drop=True)
    else:
        meta_df = pd.DataFrame()

    return pair_models, pair_stats_df, meta_df

def train_meta_model(meta_df, meta_features, target="fare_amount"):
    """
    用第一阶段各 pair 模型的验证集预测结果，训练总模型
    """
    if len(meta_df) == 0:
        raise ValueError("meta_df is empty. No pair models were trained successfully.")

    X_meta = meta_df[meta_features].fillna(0)
    y_meta = meta_df[target]

    # 再按时间切一次，保持 time-series 逻辑
    split_idx = int(len(meta_df) * 0.8)
    X_tr = X_meta.iloc[:split_idx]
    y_tr = y_meta.iloc[:split_idx]

    X_va = X_meta.iloc[split_idx:]
    y_va = y_meta.iloc[split_idx:]

    meta_model = fit_xgb_model(X_tr, y_tr)
    pred_va = meta_model.predict(X_va)

    print(f"Meta Model RMSE: {rmse(y_va.to_numpy(), pred_va):.4f}")
    print(f"Meta Model MAE : {mean_absolute_error(y_va.to_numpy(), pred_va):.4f}")

    return meta_model

In [ ]:
train_feat.keys()

Index(['key', 'fare_amount', 'pickup_datetime', 'pickup_longitude',
       'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude',
       'passenger_count', 'pu_grid_i', 'pu_grid_j', 'pu_cell_id', 'do_grid_i',
       'do_grid_j', 'do_cell_id', 'pair_id', 'pickup_hour', 'pickup_dow',
       'pickup_month', 'pickup_year', 'year_index', 'year_month_index',
       'is_weekend', 'is_rush_hour', 'is_night', 'haversine_km',
       'manhattan_km', 'distance_ratio_manhattan_haversine', 'trip_bearing',
       'bearing_sin', 'bearing_cos', 'pickup_to_jfk_km', 'dropoff_to_jfk_km',
       'pickup_to_lga_km', 'dropoff_to_lga_km', 'pickup_to_manhattan_km',
       'dropoff_to_manhattan_km', 'is_jfk_trip', 'is_lga_trip',
       'pickup_in_manhattan_core', 'dropoff_in_manhattan_core', 'dist_x_jfk',
       'dist_x_lga', 'is_short_trip', 'is_long_trip', 'dist_x_long_trip',
       'dist_x_rush', 'dist_x_night', 'hour_sin', 'hour_cos'],
      dtype='str')

In [ ]:
TRIP_THRESHOLDS = [5, 12]

df = train_feat.copy()

for t in TRIP_THRESHOLDS:
    long_col = f"is_long_trip_{t}km"
    excess_col = f"excess_dist_over_{t}km"

    df[long_col] = (df["manhattan_km"] >= t).astype("int8")
    df[excess_col] = (df["manhattan_km"] - t).clip(lower=0)

    df[f"{long_col}_x_rush"] = df[long_col] * df["is_rush_hour"]
    df[f"{long_col}_x_night"] = df[long_col] * df["is_night"]

    df[f"{long_col}_x_jfk"] = df[long_col] * df["is_jfk_trip"]
    df[f"{long_col}_x_lga"] = df[long_col] * df["is_lga_trip"]

    df[f"{long_col}_x_cell_id"] = df[long_col] * df["pu_cell_id"]
    df[f"{long_col}_x_cell_id"] = df[long_col] * df["do_cell_id"]

In [ ]:
# features:
base_features = ["passenger_count", "pickup_hour", "pickup_dow", "is_weekend", "is_night", "haversine_km"]

grid_features = ["pu_grid_i","pu_grid_j","do_grid_i","do_grid_j"]

manhattan_features = ["manhattan_km", "distance_ratio_manhattan_haversine"]

bearing_features = ["trip_bearing", "bearing_sin", "bearing_cos"]

poi_features = ["pickup_to_jfk_km", "dropoff_to_jfk_km",
    "pickup_to_lga_km", "dropoff_to_lga_km", "pickup_to_manhattan_km",
    "dropoff_to_manhattan_km", "is_jfk_trip","is_lga_trip"]

interaction_features = ["dist_x_jfk", "dist_x_lga", "dist_x_rush", "dist_x_night"]

trip_length_features = ["is_long_trip", "dist_x_long_trip"]

year_features = ["pickup_month", "pickup_year", "year_index", "year_month_index"]

core_features = ["pickup_in_manhattan_core", "dropoff_in_manhattan_core"]

FULL = base_features + grid_features + manhattan_features + poi_features + interaction_features + trip_length_features + bearing_features + year_features + core_features

PAIR_FEATURES = FULL.copy()

META_FEATURES = FULL.copy() + ["pair_pred", "pair_id"]

In [ ]:
df.head()

,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pu_grid_i,pu_grid_j,...,is_long_trip_5km_x_jfk,is_long_trip_5km_x_lga,is_long_trip_5km_x_cell_id,is_long_trip_12km,excess_dist_over_12km,is_long_trip_12km_x_rush,is_long_trip_12km_x_night,is_long_trip_12km_x_jfk,is_long_trip_12km_x_lga,is_long_trip_12km_x_cell_id
0,2009-01-01 00:00:27.0000001,30.2,2009-01-01 00:00:27+00:00,-73.782104,40.644881,-73.963565,40.676348,1,2,8,...,1,0,35,1,6.801686,0,1,1,0,35
1,2009-01-01 00:00:46.0000002,15.0,2009-01-01 00:00:46+00:00,-73.953738,40.806762,-73.989427,40.769542,1,6,5,...,0,0,55,0,0.000000,0,0,0,0,0
2,2009-01-01 00:00:49.0000002,4.2,2009-01-01 00:00:49+00:00,-73.993185,40.727841,-73.990781,40.730007,1,4,5,...,0,0,0,0,0.000000,0,0,0,0,0
3,2009-01-01 00:01:04.0000003,14.6,2009-01-01 00:01:04+00:00,-73.972484,40.742743,-73.918937,40.764496,1,4,5,...,0,0,56,0,0.000000,0,0,0,0,0
4,2009-01-01 00:01:04.0000001,5.8,2009-01-01 00:01:04+00:00,-73.995133,40.734111,-73.998232,40.722874,2,4,5,...,0,0,0,0,0.000000,0,0,0,0,0


Test XGBoost:

In [ ]:
pair_models, pair_stats_df, meta_df = train_pair_models(
    train_df=df,
    pair_features=PAIR_FEATURES,
    target="fare_amount",
    min_samples_per_pair=MIN_SAMPLES_PER_PAIR)

print("Trained pair models:", len(pair_models))
print(pair_stats_df.head())
print(meta_df.shape)

NameError: name 'train_pair_models' is not defined

In [ ]:
meta_model = train_meta_model(
    meta_df=meta_df,
    meta_features=META_FEATURES,
    target="fare_amount")

Use device = "cuda" to get the result， Optimization.

XCB FULL Features RMSE: 2.9252
XCB FULL Features MAE : 1.3877

Catboost Test:

CatBoost RMSE: 2.9567
CatBoost MAE : 1.4206